In [2]:
from pathlib import Path
import pandas as pd

# =========================
# CONFIG
# =========================

RAW_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/A.Raw Dataset")
CLEAN_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/B.Clean Dataset")

ACTIVITIES = ["Bungkuk", "Duduk", "Jongkok", "Jatuh"]

SUBJECTS = [
    "Adelia",
    "Afi",
    "Aswangga",
    "Bustan",
    "Dilia",
    "Eldivo",
    "Fathir",
    "Lina",
    "Manda",
    "Miftah",
    "Teguh",
    "Tsamara",
]

RAW_FILES = [f"{i}.Csv" for i in range(1, 10)]
CLEAN_FILES = [f"{i}.csv" for i in range(1, 10)]

SELECTED_COLUMNS = ["Timestamp", "X", "Y", "Z", "Reflectivity"]

expected_total = len(ACTIVITIES) * len(SUBJECTS) * len(RAW_FILES)

print("Raw base directory   :", RAW_BASE_DIR)
print("Clean base directory :", CLEAN_BASE_DIR)
print("Raw base exists      :", RAW_BASE_DIR.exists())
print("Selected columns     :", SELECTED_COLUMNS)
print("Expected total files :", expected_total)
print("Output extension     : lowercase .csv")

Raw base directory   : /media/spell/Spell-lab/Lidar/A.Raw Dataset
Clean base directory : /media/spell/Spell-lab/Lidar/B.Clean Dataset
Raw base exists      : True
Selected columns     : ['Timestamp', 'X', 'Y', 'Z', 'Reflectivity']
Expected total files : 432
Output extension     : lowercase .csv


In [3]:
# =========================
# GENERATE CLEAN CSV FILES
# =========================

records = []

for activity in ACTIVITIES:
    for subject in SUBJECTS:
        raw_csv_dir = RAW_BASE_DIR / activity / subject / "CSV"
        clean_subject_dir = CLEAN_BASE_DIR / activity / subject
        clean_subject_dir.mkdir(parents=True, exist_ok=True)

        for raw_name, clean_name in zip(RAW_FILES, CLEAN_FILES):
            input_path = raw_csv_dir / raw_name
            output_path = clean_subject_dir / clean_name

            status = {
                "activity": activity,
                "subject": subject,
                "raw_file": raw_name,
                "clean_file": clean_name,
                "input_path": str(input_path),
                "output_path": str(output_path),
                "input_exists": input_path.exists(),
                "success": False,
                "rows_before": None,
                "rows_after": None,
                "dropped_rows": None,
                "missing_columns": None,
                "error": None,
            }

            try:
                if not input_path.exists():
                    status["error"] = "Input file not found"
                    records.append(status)
                    continue

                # Baca raw CSV
                df = pd.read_csv(input_path)

                # Cek kolom wajib
                missing_cols = [col for col in SELECTED_COLUMNS if col not in df.columns]
                status["missing_columns"] = missing_cols

                if missing_cols:
                    status["error"] = f"Missing columns: {missing_cols}"
                    records.append(status)
                    continue

                status["rows_before"] = len(df)

                # Ambil hanya kolom penting
                df_clean = df[SELECTED_COLUMNS].copy()

                # Convert semua kolom menjadi numeric
                for col in SELECTED_COLUMNS:
                    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

                before_drop = len(df_clean)

                # Drop baris invalid
                df_clean = df_clean.dropna(subset=SELECTED_COLUMNS).copy()

                # Sort timestamp agar urutan packet/frame aman
                df_clean = df_clean.sort_values("Timestamp").reset_index(drop=True)

                after_drop = len(df_clean)

                status["rows_after"] = after_drop
                status["dropped_rows"] = before_drop - after_drop

                # Simpan sebagai pure CSV standar
                df_clean.to_csv(
                    output_path,
                    index=False,
                    encoding="utf-8",
                    sep=",",
                    lineterminator="\n"
                )

                status["success"] = True

            except Exception as e:
                status["error"] = str(e)

            records.append(status)

clean_generation_df = pd.DataFrame(records)

print("===== CLEAN DATASET GENERATION SUMMARY =====")
print("Expected files :", expected_total)
print("Success files  :", int(clean_generation_df["success"].sum()))
print("Failed files   :", int((~clean_generation_df["success"]).sum()))

print("\nDropped rows summary:")
display(clean_generation_df["dropped_rows"].describe())

print("\nFailed files preview:")
display(clean_generation_df[clean_generation_df["success"] == False].head(30))

/tmp/ipykernel_139839/839850515.py:40: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)
/tmp/ipykernel_139839/839850515.py:40: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)
/tmp/ipykernel_139839/839850515.py:40: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)
/tmp/ipykernel_139839/839850515.py:40: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)
/tmp/ipykernel_139839/839850515.py:40: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)
/tmp/ipykernel_139839/839850515.py:40: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.r

===== CLEAN DATASET GENERATION SUMMARY =====
Expected files : 432
Success files  : 432
Failed files   : 0

Dropped rows summary:


count    432.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: dropped_rows, dtype: float64


Failed files preview:


,activity,subject,raw_file,clean_file,input_path,output_path,input_exists,success,rows_before,rows_after,dropped_rows,missing_columns,error


In [4]:
# =========================
# VALIDATE CLEAN DATASET OUTPUT
# =========================

validation_records = []

for activity in ACTIVITIES:
    for subject in SUBJECTS:
        clean_subject_dir = CLEAN_BASE_DIR / activity / subject

        for clean_name in CLEAN_FILES:
            output_path = clean_subject_dir / clean_name

            rec = {
                "activity": activity,
                "subject": subject,
                "clean_file": clean_name,
                "output_path": str(output_path),
                "exists": output_path.exists(),
                "can_read": False,
                "is_lowercase_csv": output_path.name.endswith(".csv"),
                "file_size_bytes": None,
                "num_rows": None,
                "columns": None,
                "columns_exact_match": False,
                "has_na": None,
                "error": None,
            }

            try:
                if not output_path.exists():
                    rec["error"] = "Output file not found"
                    validation_records.append(rec)
                    continue

                rec["file_size_bytes"] = output_path.stat().st_size

                df_check = pd.read_csv(output_path)

                rec["can_read"] = True
                rec["num_rows"] = len(df_check)
                rec["columns"] = list(df_check.columns)
                rec["columns_exact_match"] = list(df_check.columns) == SELECTED_COLUMNS
                rec["has_na"] = bool(df_check[SELECTED_COLUMNS].isna().any().any())

            except Exception as e:
                rec["error"] = str(e)

            validation_records.append(rec)

clean_validation_df = pd.DataFrame(validation_records)

problem_validation_df = clean_validation_df[
    (clean_validation_df["exists"] == False) |
    (clean_validation_df["can_read"] == False) |
    (clean_validation_df["is_lowercase_csv"] == False) |
    (clean_validation_df["columns_exact_match"] == False) |
    (clean_validation_df["has_na"] == True)
]

print("===== CLEAN DATASET VALIDATION SUMMARY =====")
print("Expected clean files :", expected_total)
print("Existing files       :", int(clean_validation_df["exists"].sum()))
print("Readable files       :", int(clean_validation_df["can_read"].sum()))
print("Lowercase .csv files :", int(clean_validation_df["is_lowercase_csv"].sum()))
print("Exact column match   :", int(clean_validation_df["columns_exact_match"].sum()))
print("Files with NA        :", int((clean_validation_df["has_na"] == True).sum()))
print("Problem files        :", len(problem_validation_df))

print("\nProblem files preview:")
display(problem_validation_df.head(30))

print("\nRow count statistics:")
display(clean_validation_df["num_rows"].describe())

===== CLEAN DATASET VALIDATION SUMMARY =====
Expected clean files : 432
Existing files       : 432
Readable files       : 432
Lowercase .csv files : 432
Exact column match   : 432
Files with NA        : 0
Problem files        : 0

Problem files preview:


,activity,subject,clean_file,output_path,exists,can_read,is_lowercase_csv,file_size_bytes,num_rows,columns,columns_exact_match,has_na,error



Row count statistics:


count    4.320000e+02
mean     1.236300e+06
std      1.435009e+05
min      8.843520e+05
25%      1.137312e+06
50%      1.210800e+06
75%      1.312680e+06
max      2.270784e+06
Name: num_rows, dtype: float64

In [5]:
# =========================
# SUMMARY AND SAVE REPORTS
# =========================

summary_by_subject = (
    clean_validation_df
    .groupby(["activity", "subject"])
    .agg(
        expected_files=("clean_file", "count"),
        existing_files=("exists", "sum"),
        readable_files=("can_read", "sum"),
        lowercase_csv_files=("is_lowercase_csv", "sum"),
        exact_column_files=("columns_exact_match", "sum"),
        files_with_na=("has_na", lambda x: int((x == True).sum())),
        total_rows=("num_rows", "sum"),
    )
    .reset_index()
)

summary_by_subject["missing_files"] = summary_by_subject["expected_files"] - summary_by_subject["existing_files"]
summary_by_subject["completion_percent"] = 100 * summary_by_subject["existing_files"] / summary_by_subject["expected_files"]

summary_by_activity = (
    clean_validation_df
    .groupby("activity")
    .agg(
        expected_files=("clean_file", "count"),
        existing_files=("exists", "sum"),
        readable_files=("can_read", "sum"),
        exact_column_files=("columns_exact_match", "sum"),
        total_rows=("num_rows", "sum"),
    )
    .reset_index()
)

REPORT_DIR = CLEAN_BASE_DIR / "_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

generation_report_path = REPORT_DIR / "clean_generation_report.csv"
validation_report_path = REPORT_DIR / "clean_validation_report.csv"
summary_subject_path = REPORT_DIR / "clean_summary_by_subject.csv"
summary_activity_path = REPORT_DIR / "clean_summary_by_activity.csv"
problem_report_path = REPORT_DIR / "clean_problem_files.csv"

clean_generation_df.to_csv(generation_report_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
clean_validation_df.to_csv(validation_report_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
summary_by_subject.to_csv(summary_subject_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
summary_by_activity.to_csv(summary_activity_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
problem_validation_df.to_csv(problem_report_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")

print("===== SUMMARY BY ACTIVITY =====")
display(summary_by_activity)

print("\n===== SUMMARY BY SUBJECT =====")
display(summary_by_subject)

print("\n===== REPORT SAVED =====")
print("Generation report :", generation_report_path)
print("Validation report :", validation_report_path)
print("Summary subject   :", summary_subject_path)
print("Summary activity  :", summary_activity_path)
print("Problem report    :", problem_report_path)

print("\nFinal status:")
print("Clean success files :", int(clean_generation_df["success"].sum()), "/", expected_total)
print("Validation problems :", len(problem_validation_df))

===== SUMMARY BY ACTIVITY =====


,activity,expected_files,existing_files,readable_files,exact_column_files,total_rows
0,Bungkuk,108,108,108,108,131256672
1,Duduk,108,108,108,108,129272160
2,Jatuh,108,108,108,108,139663200
3,Jongkok,108,108,108,108,133889664



===== SUMMARY BY SUBJECT =====


,activity,subject,expected_files,existing_files,readable_files,lowercase_csv_files,exact_column_files,files_with_na,total_rows,missing_files,completion_percent
0,Bungkuk,Adelia,9,9,9,9,9,0,10475232,0,100.0
1,Bungkuk,Afi,9,9,9,9,9,0,10563936,0,100.0
2,Bungkuk,Aswangga,9,9,9,9,9,0,10431168,0,100.0
3,Bungkuk,Bustan,9,9,9,9,9,0,10030368,0,100.0
4,Bungkuk,Dilia,9,9,9,9,9,0,12014592,0,100.0
5,Bungkuk,Eldivo,9,9,9,9,9,0,11176416,0,100.0
6,Bungkuk,Fathir,9,9,9,9,9,0,10304256,0,100.0
7,Bungkuk,Lina,9,9,9,9,9,0,11184192,0,100.0
8,Bungkuk,Manda,9,9,9,9,9,0,11324832,0,100.0
9,Bungkuk,Miftah,9,9,9,9,9,0,10510464,0,100.0



===== REPORT SAVED =====
Generation report : /media/spell/Spell-lab/Lidar/B.Clean Dataset/_reports/clean_generation_report.csv
Validation report : /media/spell/Spell-lab/Lidar/B.Clean Dataset/_reports/clean_validation_report.csv
Summary subject   : /media/spell/Spell-lab/Lidar/B.Clean Dataset/_reports/clean_summary_by_subject.csv
Summary activity  : /media/spell/Spell-lab/Lidar/B.Clean Dataset/_reports/clean_summary_by_activity.csv
Problem report    : /media/spell/Spell-lab/Lidar/B.Clean Dataset/_reports/clean_problem_files.csv

Final status:
Clean success files : 432 / 432
Validation problems : 0
